# Dendritic vs Point Neuron

A walkthrough comparing the `DendriticNeuron` in this repo against the `PointNeuron`
baseline, on the same data, with the same split and the same hyperparameters.

Everything here is built from the scratch implementation in `src/`. There is no
PyTorch or autograd involved, so it is worth reading the forward and backward
passes alongside this notebook.

> Run all cells from the project root so that `src` and `experiments` are importable
> (`pip install -e .` also works).

In [ ]:
import sys
from pathlib import Path

# Make the project importable when running from a notebook in notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from experiments.common import build_branch_input_map, build_dataset, train_val_split
from experiments.synthetic_dataset import get_branch_index_groups
from src import DendriticNeuron, PointNeuron, get_loss_function, train_model

%matplotlib inline

## 1. The dataset

Six input features grouped into three two-feature blocks. Class 0 and class 1 are
drawn from mirrored Gaussian centres, so no single straight line separates them:

In [ ]:
INPUT_DIM = 6
NUM_SAMPLES = 1500
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 0.05
RANDOM_SEED = 42

X, y = build_dataset(num_samples=NUM_SAMPLES, input_dim=INPUT_DIM, seed=RANDOM_SEED)

print(f"dataset: {len(X)} samples x {len(X[0])} features")
print(f"class balance: {np.bincount(np.asarray(y))}")
print(f"branch groups: {get_branch_index_groups(INPUT_DIM)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (d1, d2) in zip(axes, [(0, 1), (2, 3), (4, 5)]):
    for label, color, name in [(0, "tab:blue", "Class 0"), (1, "tab:orange", "Class 1")]:
        ax.scatter(
            [row[d1] for row, lab in zip(X, y) if lab == label],
            [row[d2] for row, lab in zip(X, y) if lab == label],
            s=6, alpha=0.5, color=color, label=name,
        )
    ax.set_title(f"features {d1} vs {d2}")
axes[0].legend()
fig.suptitle("The three branch blocks: each is mirrored between the two classes")
plt.tight_layout()
plt.show()

Notice that the third block (`features 4 and 5`) is centred on the origin for both
classes, so it carries almost no signal. That is deliberate: a model that leans on
it is relying on noise.

## 2. A shared split

Both models use the identical split. This matters more than it sounds: the original
code built the dendritic split with scikit-learn and the point split with a local
helper, so the two models were never actually comparable.

In [ ]:
X_train, y_train, X_val, y_val = train_val_split(X, y, seed=RANDOM_SEED)
print(f"train: {len(X_train)}  val: {len(X_val)}")
print(f"train class balance: {np.bincount(np.asarray(y_train, dtype=int))}")
print(f"val   class balance: {np.bincount(np.asarray(y_val, dtype=int))}")

## 3. Two models, same objective

The dendritic neuron gets three branches of two features each, `tanh` branches and a
`sigmoid` soma. The point neuron gets a single weight vector with a `sigmoid`
activation. Both minimise binary cross-entropy with plain mini-batch gradient descent.

In [ ]:
loss_fn, loss_deriv_fn = get_loss_function("binary_cross_entropy")

branch_input_map = build_branch_input_map()
print("branch map:", branch_input_map)

In [ ]:
dendritic = DendriticNeuron(
    input_dim=INPUT_DIM,
    num_branches=len(branch_input_map),
    branch_input_map=branch_input_map,
    branch_activation="tanh",
    soma_activation="sigmoid",
    seed=RANDOM_SEED,
)
print(dendritic.summary())

In [ ]:
point = PointNeuron(input_dim=INPUT_DIM, activation="sigmoid", seed=RANDOM_SEED)
print(point.summary())
print("\nparameter counts:")
d_count = sum(len(w) for w in dendritic.branch_weights) + len(dendritic.branch_biases) \
    + len(dendritic.soma_weights) + 1
p_count = len(point.weights) + 1
print(f"  dendritic: {d_count}")
print(f"  point:     {p_count}")

The dendritic neuron has 16 parameters against the point neuron's 7. If it does not
win, extra capacity alone is not buying anything - which is itself the interesting
result at this scale.

In [ ]:
import random

print("=== Dendritic ===")
d_history, dendritic = train_model(
    dendritic, X_train, y_train, X_val, y_val,
    loss_fn, loss_deriv_fn,
    learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS, rng=random.Random(RANDOM_SEED),
)

In [ ]:
print("\n=== Point ===")
p_history, point = train_model(
    point, X_train, y_train, X_val, y_val,
    loss_fn, loss_deriv_fn,
    learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS, rng=random.Random(RANDOM_SEED),
)

## 4. Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for history, label, color in [
    (d_history, "Dendritic", "tab:green"),
    (p_history, "Point", "tab:purple"),
]:
    axes[0].plot(history["train_loss"], label=label, color=color)
    axes[0].plot(history["val_loss"], linestyle="--", color=color)
    axes[1].plot(history["val_accuracy"], label=label, color=color)

axes[0].set_title("Loss (solid = train, dashed = val)")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("binary cross-entropy")
axes[0].legend()
axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. What did each branch learn?

The dendritic neuron's internal state is readable, which is the practical appeal of
the structure. Push one input at a time through the branches and watch the branch
pre-activations respond independently.

In [ ]:
print("Learned branch weights:")
for b, (indices, weights, bias) in enumerate(
    zip(dendritic.branch_input_map.values(), dendritic.branch_weights, dendritic.branch_biases)
):
    pretty = ", ".join(f"w{i}={w:+.3f}" for i, w in zip(indices, weights))
    print(f"  branch {b} (features {indices}): {pretty}, b={bias:+.3f}")
print(f"  soma weights: {['%+.3f' % w for w in dendritic.soma_weights]}")
print(f"  soma bias:    {dendritic.soma_bias:+.3f}")
print()
print("Point neuron weights:")
print("  " + ", ".join(f"w{i}={w:+.3f}" for i, w in enumerate(point.weights)))
print(f"  bias: {point.bias:+.3f}")

In [ ]:
probe = [0.0] * INPUT_DIM
for b, indices in branch_input_map.items():
    dendritic.forward_single(probe)
    pre = dendritic.last_branch_pre_activation
    out = dendritic.last_branch_output
    print(f"all-zero input -> branch pre-activations {['%+.3f' % v for v in pre]}")
    print(f"                    branch outputs       {['%+.3f' % v for v in out]}")
    print(f"                    soma pre-activation  {dendritic.last_soma_pre_activation:+.3f}")
    print(f"                    output               {dendritic.forward_single(probe):.3f}")
    break

print("\nsweep each feature from -3 to +3, holding the rest at 0:")
for feature in range(INPUT_DIM):
    curve = []
    for value in np.linspace(-3, 3, 60):
        row = [0.0] * INPUT_DIM
        row[feature] = value
        curve.append(dendritic.forward_single(row))
    print(f"  feature {feature}: output ranges {min(curve):.3f} .. {max(curve):.3f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
grid = np.linspace(-4, 4, 80)

for feature, ax in enumerate(axes.ravel()):
    for model, label in [(dendritic, "Dendritic"), (point, "Point")]:
        # DendriticNeuron exposes forward_single, PointNeuron exposes forward.
        single = getattr(model, "forward_single", None) or model.forward
        outs = []
        for value in grid:
            row = [0.0] * INPUT_DIM
            row[feature] = value
            outs.append(single(row))
        outs = np.array(outs)
        ax.plot(grid, outs, label=label)
        ax.fill_between(grid, 0, 1, where=outs >= 0.5, alpha=0.08, color="gray")
    ax.axhline(0.5, color="black", lw=0.6, ls=":")
    ax.set_title(f"feature {feature}")
    ax.set_xlabel("value")
    ax.set_ylabel("output")
axes[0][0].legend()
fig.suptitle("Response to each feature in isolation (shaded = predicted class 1)")
plt.tight_layout()
plt.show()

## 6. Decision region on one branch pair

Sweep the first branch's two features and hold everything else at zero, so the shape
of the boundary is driven by that one branch.

In [ ]:
def decision_grid(model, d1, d2, extent=(-4, 4), n=120, background=0.0):
    """Evaluate a model on a 2D slice, holding other features at `background`."""
    xs = np.linspace(*extent, n)
    ys = np.linspace(*extent, n)
    grid = np.zeros((n, n, INPUT_DIM))
    for i, v1 in enumerate(xs):
        for j, v2 in enumerate(ys):
            row = [background] * INPUT_DIM
            row[d1] = v1
            row[d2] = v2
            grid[i, j] = row
    flat = [list(r) for r in grid.reshape(-1, INPUT_DIM)]
    return model.forward_batch(flat), xs, ys


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for ax, (model, title) in zip(axes, [(dendritic, "Dendritic"), (point, "Point")]):
    probs, xs, ys = decision_grid(model, 0, 1)
    probs = np.asarray(probs).reshape(len(xs), len(ys))
    ax.contourf(xs, ys, probs, levels=20, cmap="RdBu_r", alpha=0.7)
    ax.contour(xs, ys, probs, levels=[0.5], colors="k", linewidths=1.5)
    for label, color in [(0, "tab:blue"), (1, "tab:orange")]:
        ax.scatter(
            [r[0] for r, lab in zip(X, y) if lab == label],
            [r[1] for r, lab in zip(X, y) if lab == label],
            s=4, alpha=0.35, color=color, edgecolors="none",
        )
    ax.set_title(f"{title}: boundary on features 0 and 1")
    ax.set_xlabel("feature 0")
    ax.set_ylabel("feature 1")
plt.tight_layout()
plt.show()

## 7. Robustness to input noise

In [ ]:
print(f"{'noise std':>10} | {'dendritic acc':>14} | {'point acc':>10} | {'dendritic loss':>15} | {'point loss':>11}")
print("-" * 70)
for noise_std in [0.0, 0.25, 0.5, 1.0, 2.0]:
    noisy = (
        np.asarray(X, dtype=float)
        + np.random.default_rng(7).normal(0.0, noise_std, size=(len(X), INPUT_DIM))
    ).tolist()
    row = []
    for model in (dendritic, point):
        preds = model.forward_batch(noisy)
        preds_label = (np.asarray(preds) >= 0.5).astype(int)
        acc = float(np.mean(preds_label == np.asarray(y)))
        row.append((acc, loss_fn(preds, y)))
    print(
        f"{noise_std:>10.2f} | {row[0][0]:>14.4f} | {row[1][0]:>10.4f} | "
        f"{row[0][1]:>15.4f} | {row[1][1]:>11.4f}"
    )

## 8. Summary

In [ ]:
print(f"{'model':>10} | {'params':>7} | {'train loss':>11} | {'val loss':>9} | {'val acc':>8}")
print("-" * 54)
for name, history, count in [("Dendritic", d_history, d_count), ("Point", p_history, p_count)]:
    print(
        f"{name:>10} | {count:>7} | {history['train_loss'][-1]:>11.4f} | "
        f"{history['val_loss'][-1]:>9.4f} | {history['val_accuracy'][-1]:>8.4f}"
    )

### Where to go next

* Vary the number of branches and the branch-to-feature mapping to see what the
  structure is actually contributing.
* Swap the branch nonlinearity per branch (`relu` vs `tanh` vs `linear`).
* Drop the noise down and watch the boundary become piecewise in the dendritic case
  and globally linear in the point case.